# Feature correlation and preprocessing sensitivity

Assess predictor correlations, variance inflation factors, Logistic Regression coefficient stability, and preprocessing alternatives. Sensitivity analyses retain hyperparameters from the primary GridSearch.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

DATA_FILE = PROJECT_ROOT / "data" / "gallstone.xlsx"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Download the UCI Gallstone dataset and save it to {DATA_FILE}")


In [ ]:
"""Correlation, multicollinearity, and preprocessing sensitivity analysis.

Scaling is fitted within each training fold; the primary held-out split is fixed.
"""
from pathlib import Path
import json, time, sys, scipy, sklearn, xgboost
import numpy as np
from scipy.stats import rankdata
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef

from zipfile import ZipFile, ZIP_DEFLATED
from xml.etree import ElementTree as ET
from xml.sax.saxutils import escape
import re

_NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'

def read_xlsx_rows(path):
    """Read the first worksheet of the supplied UCI XLSX with only standard Python."""
    with ZipFile(path) as z:
        shared = []
        if 'xl/sharedStrings.xml' in z.namelist():
            root = ET.fromstring(z.read('xl/sharedStrings.xml'))
            for si in root.findall(_NS+'si'):
                shared.append(''.join(t.text or '' for t in si.iter(_NS+'t')))
        ws = ET.fromstring(z.read('xl/worksheets/sheet1.xml'))
        lines = []
        for row in ws.findall('.//'+_NS+'sheetData/'+_NS+'row'):
            fields = {}
            for cell in row.findall(_NS+'c'):
                letter = re.match(r'[A-Z]+',cell.attrib['r']).group()
                col = 0
                for ch in letter:
                    col = col*26 + ord(ch)-64
                v = cell.find(_NS+'v')
                ctype = cell.attrib.get('t')
                if ctype == 's' and v is not None:
                    value = shared[int(v.text)]
                elif ctype == 'inlineStr':
                    x = cell.find(_NS+'is')
                    value = ''.join(n.text or '' for n in x.iter(_NS+'t')) if x is not None else ''
                elif v is None:
                    value = None
                elif ctype in ('str','e'):
                    value = v.text
                else:
                    try: value = float(v.text)
                    except (ValueError,TypeError): value = v.text
                fields[col-1] = value
            lines.append([fields.get(i) for i in range(max(fields.keys(), default=-1)+1)])
    return lines

def _col(i):
    name='';i+=1
    while i:
        i,r=divmod(i-1,26)
        name=chr(65+r)+name
    return name

def write_report_xlsx(path, sheets):
    """Write one valid multi-sheet XLSX with no extra spreadsheet packages."""
    content_types = ['<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">',
        '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>',
        '<Default Extension="xml" ContentType="application/xml"/>',
        '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>',
        '<Override PartName="/xl/styles.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.styles+xml"/>']
    for n in range(1,len(sheets)+1):
        content_types.append(f'<Override PartName="/xl/worksheets/sheet{n}.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>')
    content_types.append('</Types>')
    workbook = ('<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships"><sheets>'
        + ''.join(f'<sheet name="{escape(k)}" sheetId="{i}" r:id="rId{i}"/>' for i,k in enumerate(sheets,1))+'</sheets></workbook>')
    wb_rels = ('<?xml version="1.0" encoding="UTF-8"?>'
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
        + ''.join(f'<Relationship Id="rId{i}" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet{i}.xml"/>' for i in range(1,len(sheets)+1))
        + f'<Relationship Id="rId{len(sheets)+1}" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/styles" Target="styles.xml"/></Relationships>')
    styles = ('<?xml version="1.0" encoding="UTF-8"?>'
        '<styleSheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
        '<fonts count="2"><font><sz val="10"/><name val="Arial"/></font><font><b/><color rgb="FFFFFFFF"/><sz val="10"/><name val="Arial"/></font></fonts>'
        '<fills count="3"><fill><patternFill patternType="none"/></fill><fill><patternFill patternType="gray125"/></fill><fill><patternFill patternType="solid"><fgColor rgb="FF16324F"/><bgColor indexed="64"/></patternFill></fill></fills>'
        '<borders count="1"><border><left/><right/><top/><bottom/><diagonal/></border></borders>'
        '<cellStyleXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0"/></cellStyleXfs>'
        '<cellXfs count="3"><xf numFmtId="0" fontId="0" fillId="0" borderId="0" xfId="0"/><xf numFmtId="0" fontId="1" fillId="2" borderId="0" xfId="0" applyFill="1" applyFont="1"/><xf numFmtId="2" fontId="0" fillId="0" borderId="0" xfId="0" applyNumberFormat="1"/></cellXfs></styleSheet>')
    with ZipFile(path,'w',compression=ZIP_DEFLATED) as z:
        z.writestr('[Content_Types].xml', ''.join(content_types))
        z.writestr('_rels/.rels', '<?xml version="1.0" encoding="UTF-8"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/></Relationships>')
        z.writestr('xl/workbook.xml',workbook)
        z.writestr('xl/_rels/workbook.xml.rels',wb_rels)
        z.writestr('xl/styles.xml',styles)
        for n,(name,(header,rows,widths)) in enumerate(sheets.items(),1):
            xml = ['<?xml version="1.0" encoding="UTF-8"?><worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">']
            xml.append(f'<dimension ref="A1:{_col(len(header)-1)}{len(rows)+1}"/>')
            xml.append('<sheetViews><sheetView workbookViewId="0"><pane ySplit="1" topLeftCell="A2" activePane="bottomLeft" state="frozen"/></sheetView></sheetViews>')
            xml.append('<cols>')
            for j in range(len(header)):
                wd = widths[j] if widths and j<len(widths) else (30 if j<2 else 20)
                xml.append(f'<col min="{j+1}" max="{j+1}" width="{max(10,min(wd,85))}" customWidth="1"/>')
            xml.append('</cols><sheetData>')
            for ri,line in enumerate([header]+rows,1):
                xml.append(f'<row r="{ri}">')
                for j,v in enumerate(line):
                    if v is None:continue
                    addr=f'{_col(j)}{ri}'
                    if isinstance(v,(int,np.integer)) and not isinstance(v,bool):
                        xml.append(f'<c r="{addr}" s="{1 if ri==1 else 0}"><v>{int(v)}</v></c>')
                    elif isinstance(v,(float,np.floating)) and np.isfinite(v):
                        xml.append(f'<c r="{addr}" s="{1 if ri==1 else 2}"><v>{repr(float(v))}</v></c>')
                    else:
                        xml.append(f'<c r="{addr}" s="{1 if ri==1 else 0}" t="inlineStr"><is><t>{escape(str(v))}</t></is></c>')
                xml.append('</row>')
            xml.append('</sheetData>')
            xml.append(f'<autoFilter ref="A1:{_col(len(header)-1)}{len(rows)+1}"/></worksheet>')
            z.writestr(f'xl/worksheets/sheet{n}.xml',''.join(xml))

DATA_PATH = DATA_FILE
OUT_PATH = RESULTS_DIR / "correlation_preprocessing.xlsx"
SEED=42
BINARY_RAW_POSITIVE=0
FEATURE_GROUPS={
 'Demographic':['Age','Gender','Height','Weight','Body Mass Index (BMI)'],
 'Clinical':['Comorbidity','Coronary Artery Disease (CAD)','Hypothyroidism','Hyperlipidemia','Diabetes Mellitus (DM)','Hepatic Fat Accumulation (HFA)'],
 'Bioimpedance':['Total Body Water (TBW)','Extracellular Water (ECW)','Intracellular Water (ICW)','Extracellular Fluid/Total Body Water (ECF/TBW)','Total Body Fat Ratio (TBFR) (%)','Lean Mass (LM) (%)','Body Protein Content (Protein) (%)','Visceral Fat Rating (VFR)','Bone Mass (BM)','Muscle Mass (MM)','Obesity (%)','Total Fat Content (TFC)','Visceral Fat Area (VFA)','Visceral Muscle Area (VMA) (Kg)'],
 'Laboratory':['Glucose','Total Cholesterol (TC)','Low Density Lipoprotein (LDL)','High Density Lipoprotein (HDL)','Triglyceride','Aspartat Aminotransferaz (AST)','Alanin Aminotransferaz (ALT)','Alkaline Phosphatase (ALP)','Creatinine','Glomerular Filtration Rate (GFR)','C-Reactive Protein (CRP)','Hemoglobin (HGB)','Vitamin D']}
SCENARIOS={
 'A_baseline': FEATURE_GROUPS['Demographic']+FEATURE_GROUPS['Clinical'],
 'B_plus_bia': FEATURE_GROUPS['Demographic']+FEATURE_GROUPS['Clinical']+FEATURE_GROUPS['Bioimpedance'],
 'C_plus_lab': FEATURE_GROUPS['Demographic']+FEATURE_GROUPS['Clinical']+FEATURE_GROUPS['Laboratory'],
 'D_full': sum(FEATURE_GROUPS.values(),[])}


PARAMS_JSON='''{"A_baseline|LR": {"clf__C": 0.8497534359086438}, "A_baseline|SVM": {"clf__C": 12.915496650148826, "clf__gamma": 0.12915496650148828}, "A_baseline|RF": {"clf__max_depth": 5, "clf__min_samples_leaf": 1, "clf__min_samples_split": 2, "clf__n_estimators": 50}, "A_baseline|XGB": {"clf__learning_rate": 0.1, "clf__max_depth": 3, "clf__n_estimators": 50}, "B_plus_bia|LR": {"clf__C": 2.420128264794381}, "B_plus_bia|SVM": {"clf__C": 12.915496650148826, "clf__gamma": 0.016681005372000592}, "B_plus_bia|RF": {"clf__max_depth": 5, "clf__min_samples_leaf": 2, "clf__min_samples_split": 2, "clf__n_estimators": 225}, "B_plus_bia|XGB": {"clf__learning_rate": 0.05, "clf__max_depth": 3, "clf__n_estimators": 100}, "C_plus_lab|LR": {"clf__C": 1.9179102616724888}, "C_plus_lab|SVM": {"clf__C": 4.6415888336127775, "clf__gamma": 0.016681005372000592}, "C_plus_lab|RF": {"clf__max_depth": 15, "clf__min_samples_leaf": 1, "clf__min_samples_split": 5, "clf__n_estimators": 100}, "C_plus_lab|XGB": {"clf__learning_rate": 0.1, "clf__max_depth": 3, "clf__n_estimators": 50}, "D_full|LR": {"clf__C": 2.1544346900318843}, "D_full|SVM": {"clf__C": 100.0, "clf__gamma": 0.005994842503189409}, "D_full|RF": {"clf__max_depth": 10, "clf__min_samples_leaf": 2, "clf__min_samples_split": 2, "clf__n_estimators": 225}, "D_full|XGB": {"clf__learning_rate": 0.01, "clf__max_depth": 3, "clf__n_estimators": 150}}'''
BEST_PARAMS=json.loads(PARAMS_JSON)


def get_dataset(path):
 cells=read_xlsx_rows(path)
 head=cells[0]; idx={c:i for i,c in enumerate(head) if c is not None}
 cols=sum(FEATURE_GROUPS.values(),[])
 assert len(cols)==38 and len(set(cols))==38
 assert all(c in idx for c in cols) and 'Gallstone Status' in idx
 vals=np.array([[float(row[idx[c]]) for c in cols] for row in cells[1:] if row[idx['Gallstone Status']] is not None],dtype=float)
 targets=np.array([int(float(row[idx['Gallstone Status']])) for row in cells[1:] if row[idx['Gallstone Status']] is not None])
 assert vals.shape==(319,38) and np.isfinite(vals).all()
 assert set(targets)=={0,1}
 y=(targets==BINARY_RAW_POSITIVE).astype(int)
 return vals,y,cols


def fgroup(c):
 for k,v in FEATURE_GROUPS.items():
  if c in v: return k
 raise ValueError(c)


def correlation_analysis(X,features,binary_flags):

 r=np.apply_along_axis(rankdata,0,X)
 corr=np.corrcoef(r,rowvar=False)
 pairs=[]
 for i in range(len(features)):
  for j in range(i+1,len(features)):
   a,b=features[i],features[j]
   pairs.append([a,b,fgroup(a),fgroup(b),float(corr[i,j]),abs(float(corr[i,j])), 'Within' if fgroup(a)==fgroup(b) else 'Between'])
 pairs.sort(key=lambda row:-row[5]); groups=list(FEATURE_GROUPS)
 summary=[]
 for gi,g in enumerate(groups):
  for h in groups[gi:]:
   rr=[p for p in pairs if set([p[2],p[3]])==set([g,h])]
   summary.append([g,h,len(rr),sum(p[5]>=0.7 for p in rr),sum(p[5]>=0.8 for p in rr),sum(p[5]>=0.9 for p in rr),max(p[5] for p in rr),next(p[0]+' | '+p[1] for p in rr if p[5]==max(x[5] for x in rr))])

 cont=[j for j,c in enumerate(features) if not binary_flags[j] and c not in {'Comorbidity','Hepatic Fat Accumulation (HFA)'}]
 xx=X[:,cont]
 xx=(xx-xx.mean(axis=0))/xx.std(axis=0,ddof=0)
 vif=[]
 for ind,j in enumerate(cont):
  other=np.delete(xx,ind,axis=1)
  beta=np.linalg.lstsq(other,xx[:,ind],rcond=None)[0]
  rss=float(np.sum((xx[:,ind]-other@beta)**2))
  tss=float(np.sum(xx[:,ind]**2))
  r2=1-rss/tss
  val=float('inf') if 1-r2<1e-10 else 1/(1-r2)
  vif.append([features[j],fgroup(features[j]),round(r2,7),val])
 vif.sort(key=lambda row:-row[-1])
 return pairs,summary,vif


def get_estimator(model):
 if model=='LR': return LogisticRegression(solver='lbfgs',max_iter=2000,random_state=SEED)
 if model=='SVM': return SVC(kernel='rbf',probability=False,random_state=SEED)
 if model=='RF': return RandomForestClassifier(criterion='gini',bootstrap=True,random_state=SEED,n_jobs=2)
 if model=='XGB': return XGBClassifier(objective='binary:logistic',eval_metric='logloss',random_state=SEED,n_jobs=2,verbosity=0)
 raise ValueError(model)


class SelectedColumnsScaler(BaseEstimator,TransformerMixin):
 """Scale specified continuous columns without reordering any predictors."""
 def __init__(self,indices=(),robust=False):
  self.indices=indices
  self.robust=robust
 def fit(self,X,y=None):
  self.scaler_=RobustScaler() if self.robust else StandardScaler()
  self.scaler_.fit(np.asarray(X)[:,list(self.indices)])
  return self
 def transform(self,X):
  arr=np.array(X,dtype=float,copy=True)
  arr[:,list(self.indices)]=self.scaler_.transform(arr[:,list(self.indices)])
  return arr

def preprocess(kind,binary_flags,features):
 cont=tuple(j for j,b in enumerate(binary_flags) if not b)
 if kind=='Standard_all': return StandardScaler()
 if kind=='Standard_continuous': return SelectedColumnsScaler(indices=cont,robust=False)
 if kind=='Robust_continuous': return SelectedColumnsScaler(indices=cont,robust=True)
 if kind=='Onehot_ordinal':
  ord_cols=[j for j,c in enumerate(features) if c in ['Comorbidity','Hepatic Fat Accumulation (HFA)']]
  numeric=[j for j in cont if j not in ord_cols]
  binary=[j for j,bb in enumerate(binary_flags) if bb]
  return ColumnTransformer([('continuous',StandardScaler(),numeric),('binary','passthrough',binary),('ordinal',OneHotEncoder(handle_unknown='ignore',sparse_output=False),ord_cols)],sparse_threshold=0)
 if kind=='No_scaling':return 'passthrough'
 raise ValueError(kind)


def metric_values(y,yp,scores):
 return [float(accuracy_score(y,yp)),float(f1_score(y,yp,zero_division=0)),float(roc_auc_score(y,scores)),float(balanced_accuracy_score(y,yp)),float(matthews_corrcoef(y,yp))]


def sensitivity(X,y,features,binary_flags):
 ti,vi=train_test_split(np.arange(len(y)),test_size=.20,stratify=y,random_state=SEED)
 cv=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(np.zeros(len(ti)),y[ti]))
 result=[]
 stage_start=time.perf_counter()
 for scen,fs in SCENARIOS.items():
  ix=[features.index(c) for c in fs]
  a=X[:,ix]; bf=binary_flags[ix]
  for model in ['LR','SVM','RF','XGB']:
   params=BEST_PARAMS[scen+'|'+model]
   for variant in ['Standard_all','Standard_continuous','Robust_continuous','No_scaling','Onehot_ordinal']:
    pipe=Pipeline([('preprocess',preprocess(variant,bf,fs)),('clf',get_estimator(model))])
    pipe.set_params(**params)
    scores=[]
    tic=time.perf_counter()
    for ftr,fva in cv:
     q=clone(pipe)
     q.fit(a[ti[ftr]],y[ti[ftr]])
     scores.append(float(accuracy_score(y[ti[fva]],q.predict(a[ti[fva]]))))
    q=clone(pipe);q.fit(a[ti],y[ti]);yp=q.predict(a[vi])
    score=q.predict_proba(a[vi])[:,1] if hasattr(q,'predict_proba') else q.decision_function(a[vi])
    results_row=[scen,model,variant,len(fs),int(bf.sum()),float(np.mean(scores)),float(np.std(scores,ddof=1)),*metric_values(y[vi],yp,score),round(time.perf_counter()-tic,3)]
    result.append(results_row)
    print(f"{len(result):02d}/80 {scen} {model} {variant}: CV={results_row[5]:.3f} test={results_row[7]:.3f}",flush=True)
 print(f'Sensitivity completed in {time.perf_counter()-stage_start:.1f}s',flush=True)
 return result,ti,vi


def logistic_coefficient_stability(X,y,features,train_idx):
 folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(np.zeros(len(train_idx)),y[train_idx]))
 data=[]
 for scen,group_cols in SCENARIOS.items():
  indices=[features.index(f) for f in group_cols]
  original_grid_params=BEST_PARAMS[scen+'|LR']
  coefs=[]
  for tr,va in folds:
   model=Pipeline([('preprocess',StandardScaler()),('clf',get_estimator('LR'))])
   model.set_params(**original_grid_params)
   model.fit(X[train_idx[tr]][:,indices],y[train_idx[tr]])
   coefs.append(model.named_steps['clf'].coef_[0])
  coefs=np.array(coefs)
  for i,name in enumerate(group_cols):
   v=coefs[:,i]
   data.append([scen,name,fgroup(name),float(np.mean(v)),float(np.std(v,ddof=1)),float(np.min(v)),float(np.max(v)),int(sum(v>0)),int(sum(v<0)), 'Yes' if min(v)<0<max(v) else 'No'])
 return data


def build_excel(X,y,features,bf,pairs,gstats,vif,sens,train_idx,test_idx,coef_rows):
 sheets={}
 def sheet(name,header,rows,widths=None):
  sheets[name]=(header,rows,widths)
 sheet('README',['Property','Description'],[
  ['Purpose','R1-2, R1-5, R2-3: feature-group correlations, multicollinearity and preprocessing sensitivity'],
  ['Dataset','UCI Gallstone, 319 observations, 38 predictors'],
  ['Grouping','Demographic 5; Clinical 6; Bioimpedance 14; Laboratory 13, unchanged from original'],
  ['Positive class','RAW Gallstone Status = 0 mapped to positive class 1'],
  ['Correlation','Spearman rho on full predictor dataset, descriptive only; no outcome used'],
  ['Binary criterion','Exactly two distinct observed numeric values 0 and 1'],
  ['VIF','OLS diagnostic for continuous predictors; binary/ordinal predictors excluded; no feature selection'],
  ['Sensitivity split','Original stratified 80:20 split, seed 42 (255 train, 64 test)'],
  ['CV','Original 5 folds, stratified shuffled seed 42; scaler fitted inside each training fold'],
  ['HPO handling','Same original GridSearch-selected hyperparameters reused for ALL preprocessing variants. No retuning.'],
  ['Important limitation','Sensitivity is conditional on original hyperparameters; not a head-to-head independently retuned comparison'],
  ['Variants','Standard_all (original), Standard_continuous (binary unchanged), Robust_continuous (binary unchanged), No_scaling, Onehot_ordinal (Comorbidity/HFA one hot)'],
  ['Categorical/ordinal','Comorbidity is 0-3 (count categories) and HFA 0-4 (severity grades). Ordinal-as-numeric preserved in original; one-hot explored only as sensitivity.'],
  ['Data dictionary','https://archive.ics.uci.edu/dataset/1150/gallstone-1'],
  ['Metrics','Cross-validation accuracy mean/SD; held-out accuracy, F1, ROC-AUC, balanced accuracy, MCC'],
  ['Uncertainty','Do not infer statistical significance from one held-out split or CV folds'],
  ['Software versions',f'numpy {np.__version__}; scipy {scipy.__version__}; scikit-learn {sklearn.__version__}; xgboost {xgboost.__version__}'],
  ['LR coefficients','Standardized L2 Logistic Regression coefficients across five training CV folds; sign flips are descriptive, not inferential'],
 ],[26,95])
 sheet('FeatureTypes',['Feature','Group','Number unique','Binary 0-1','Min','Max'],[[c,fgroup(c),len(np.unique(X[:,j])),'Yes' if bf[j] else 'No',float(np.min(X[:,j])),float(np.max(X[:,j]))] for j,c in enumerate(features)],[48,20,16,14,15,15])
 sheet('TopCorrelations',['Feature 1','Feature 2','Group 1','Group 2','Spearman rho','Abs rho','Location'],pairs[:60],[45,45,18,18,16,16,14])
 sheet('GroupCorrelations',['Group 1','Group 2','Pairs','Abs rho >= 0.7','Abs rho >= 0.8','Abs rho >= 0.9','Max abs rho','Strongest pair'],gstats,[23,23,13,17,17,17,17,70])
 sheet('VIF_Numerical',['Feature','Group','R squared to others','Variance Inflation Factor'],vif,[48,20,26,26])
 shead=['Scenario','Classifier','Preprocessing','N features','N binary','CV Acc mean','CV Acc SD','Test accuracy','Test F1','Test ROC-AUC','Test balanced acc','Test MCC','Seconds']
 sheet('PreprocessMetrics',shead,sens,[20,15,26,14,12,16,15,17,15,17,20,16,14])
 base={(r[0],r[1]):r for r in sens if r[2]=='Standard_all'}
 delta=[]
 for r in sens:
  orig=base[r[0],r[1]]
  delta.append([r[0],r[1],r[2],*[float(r[k]-orig[k]) for k in (5,7,8,9,10,11)]])
 sheet('SensitivityDeltas',['Scenario','Classifier','Preprocessing','Delta CV Accuracy','Delta Test Accuracy','Delta F1','Delta ROC-AUC','Delta BalAcc','Delta MCC'],delta,[20,15,25,21,22,14,17,17,17])
 sheet('LRCoefficientStability',['Scenario','Feature','Group','Coef mean','Coef SD','Coef min','Coef max','Positive folds','Negative folds','Sign reversal'],coef_rows,[20,48,19,16,16,16,16,18,18,18])
 sheet('HyperparametersUsed',['Scenario','Classifier','Original GridSearch parameters'],[[ss,md,json.dumps(BEST_PARAMS[ss+'|'+md],sort_keys=True)] for ss in SCENARIOS for md in ['LR','SVM','RF','XGB']],[20,15,82])
 sheet('SplitIndices',['Original row index','Partition'],[[i,'Train' if i in set(train_idx) else 'Test'] for i in range(len(y))],[22,16])
 write_report_xlsx(OUT_PATH,sheets)
 assert OUT_PATH.exists() and OUT_PATH.stat().st_size>5000
 return OUT_PATH

if __name__=='__main__':
 X,y,features=get_dataset(DATA_PATH)
 binary_flags=np.array([np.array_equal(np.unique(X[:,i]),np.array([0.,1.])) for i in range(X.shape[1])])
 pairs,gstats,vif=correlation_analysis(X,features,binary_flags)
 print('Dataset:',X.shape,'Binary:',binary_flags.sum())
 print('Top rho',[(p[0],p[1],round(p[4],3)) for p in pairs[:10]])
 sensitivity_results,tr,te=sensitivity(X,y,features,binary_flags)
 coef_rows=logistic_coefficient_stability(X,y,features,tr)
 out=build_excel(X,y,features,binary_flags,pairs,gstats,vif,sensitivity_results,tr,te,coef_rows)
 print('SAVED',out.resolve(),out.stat().st_size)


